# KAT-VAD v2 — **TAD zero-shot: does v2 transfer to a fixed camera?** (Amendment 15, §26 T1–T10)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§26**. A generalization probe: the T2-trained checkpoints
(A0, A3 from E3; **A3-ign** = the deployed `B` from E4-0) scored on TAD's official 100-video test protocol at
**stride 8** — Gate T0's protocol, so LaGoVAD's released checkpoint is read on the identical rows.
**No training. Nothing re-decides (T1)**: `B` = A3-ign stands whatever the numbers show.

| step | what | GPU? |
|---|---|---|
| 0 | setup; §26 must be in the upload | — |
| 1 | preflight: 15 checkpoints (J10 + arm config), TAD labels / `TAD_ncc` on Drive, the T2 A3 bake + its V2-S manifest | no |
| 2 | Kaggle frames → numeric-order symlink farm (test ids only); T9(c) frame counts | no |
| 3 | T9(c) frame identity: CLIP s8 re-extracted on 5 videos vs `TAD_ncc` | yes (small) |
| 4 | VideoMAE **V2-S** at stride 8 on the 100 test videos (T4) → Drive | yes |
| 5 | bake A3's input on TAD with the **T2** stats (T3) → Drive; T9(f) | no |
| 6 | score: T0 + {A0, A3, A3-ign} × 5 seeds with `core.evaluate` | yes |
| 7 | `tadz_readout` (T9 gates first) → **stop** | no |

Every step skips what is already on Drive. Outputs: `Thesis-V2/outputs/v2_tadz/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/TAD, cache/clip/TAD_ncc, ckpts/best.ckpt
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, cache/v2/, outputs/; write: cache/video, cache/v2, outputs/v2_tadz
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/tadz_readout.py').is_file(), 'this upload predates §26 -- re-upload branch v2'
addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 26. Amendment 15' in addendum, 'upload has no addendum §26'
s26 = addendum.split('## 26. Amendment 15', 1)[1]
assert 'Authorized by the user on' in s26 and 'Not in force until' not in s26, '§26 is not signed in this upload'

DATASET  = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig' (the training corpus)
SEEDS    = list(constants.V2_E3_SEEDS)                 # 2024-2028
STEP     = 1740
STRIDE   = constants.V2_TADZ_STRIDE                    # 8
CRN      = 'R2'
V2S      = constants.VIDEOMAE_ENCODER_S                # 'vit_s_k710_dl_from_giant'
ARMS     = list(constants.V2_TADZ_ARMS)                # A0, A3, A3ign
EXPECT   = {'A0': (constants.V2_OFF, constants.V2_OFF, constants.DVS_ANCHOR_MODE_SPAN),
            'A3': (CRN, V2S, constants.DVS_ANCHOR_MODE_SPAN),
            'A3ign': (CRN, V2S, constants.DVS_ANCHOR_MODE_IGNORE)}

TAD      = constants.DATA_ROOT / constants.TAD_DATASET            # Thesis/data/TAD (T0's labels, s8)
TAD_NCC  = constants.CLIP_CACHE_DIR / 'TAD_ncc'                   # Thesis/cache/clip/TAD_ncc (T0's features)
T0_CKPT  = Path(DRIVE) / 'ckpts' / 'best.ckpt'                    # LaGoVAD released
V2_CACHE = Path(DRIVE_V2) / 'cache'
T2_BAKE  = V2_CACHE / 'v2' / f'A3_{CRN}_S' / DATASET               # A3 / A3-ign's T2 input: the stats (T3)
TAD_V2S  = V2_CACHE / 'video' / V2S / f'{constants.TAD_DATASET}_s{STRIDE}_squash'   # written in step 4
TAD_BAKE = V2_CACHE / 'v2' / f'A3_{CRN}_S' / f'{constants.TAD_DATASET}_s{STRIDE}'    # written in step 5
V2_DATA  = Path(DRIVE_V2) / 'data' / f'{DATASET}_v2'               # p_T2's training windows (T7)
E3       = constants.OUTPUT_ROOT / 'v2_e3'
E40      = constants.OUTPUT_ROOT / 'v2_e4_0'
OUT      = constants.OUTPUT_ROOT / 'v2_tadz'
KAGGLE   = 'nikanvasei/traffic-anomaly-dataset-tad'
LOCAL    = Path('/content/tadz')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('TAD', 'TAD_NCC', 'T0_CKPT', 'T2_BAKE', 'TAD_V2S', 'TAD_BAKE', 'V2_DATA', 'OUT'):
    print(f'{name:9s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu kagglehub
python -c "import transformers, av, einops, faiss, kagglehub; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 3/4/6 will be slow'
df -h /content | tail -1

In [ ]:
import json
import math
import pickle
import re
import shutil
import subprocess
import time

import numpy as np
import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage_ids(src, dst, ids):
    '''Copy ``ids`` (+ the dir's manifests) from Drive to VM disk once (bit-identical; C2).'''
    dst.mkdir(parents=True, exist_ok=True)
    for v in ids:
        if not (dst / f'{v}.npy').exists():
            shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
    for m in list(src.glob('*.json')) + list(src.glob('*.npz')):
        shutil.copy2(m, dst / m.name)
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path} ({path.stat().st_size / 2**20:.0f} MiB): {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def run_dir(arm, seed):
    return (E40 if arm == constants.V2_TADZ_ARM else E3) / arm / f's{seed}'


def complete(dir_, ids):
    return dir_.is_dir() and all((dir_ / f'{v}.npy').exists() for v in ids)

## 1. Preflight — every input exists and is the one §26 names (T3, T9(b)(e))

In [ ]:
for seed in SEEDS:                       # T9(e): J10 + the arm's config on every checkpoint
    for arm in ARMS:
        d = run_dir(arm, seed)
        step, last = ckpt_step(d / 'checkpoint_last.pt'), metrics_last(d / 'metrics.jsonl')
        assert last is not None and step == last['global_step'] == STEP, f'{arm} s{seed}: J10 (ckpt {step})'
        cfg = yaml.safe_load((d / 'config.yaml').read_text())
        got = (cfg['v2']['crn'], cfg['v2']['motion'], cfg['loss']['dvs_anchor_mode'])
        assert got == EXPECT[arm], f'{arm} s{seed}: config {got}, expected {EXPECT[arm]}'
assert T0_CKPT.is_file(), f'LaGoVAD released checkpoint missing: {T0_CKPT}'
print(f'15 checkpoints OK at step {STEP}; T0 checkpoint present')

anno = json.loads((TAD / 'annotations' / 'tad_test_anno.json').read_text())
labels_s8 = json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())
test_ids = sorted(labels_s8)
n_abn = sum(int(max(g) > 0) for g in labels_s8.values())
assert (len(test_ids), n_abn) == (constants.V2_TADZ_VIDEOS, constants.V2_TADZ_ABNORMAL), (len(test_ids), n_abn)
assert {Path(e['video_path']).stem for e in anno} == set(test_ids), 'labels are not tad_test_anno.json'
missing = [v for v in test_ids if not (TAD_NCC / f'{v}.npy').exists()]
assert not missing, f'TAD_ncc lacks {len(missing)} test videos: {missing[:5]}'
bad = [v for v in test_ids if len(np.load(TAD_NCC / f'{v}.npy', mmap_mode='r')) != len(labels_s8[v])]
assert not bad, f'TAD_ncc rows != s8 label rows on {bad[:5]}'
(LOCAL / 'test_ids.txt').write_text('\n'.join(test_ids) + '\n')
print(f'TAD protocol: {len(test_ids)} videos, {n_abn} abnormal; TAD_ncc rows == s8 labels')

t2_bake = json.loads((T2_BAKE / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (t2_bake['crn'], t2_bake['motion']) == (CRN, V2S), t2_bake
cands = [Path(t2_bake['video_dir']) if t2_bake.get('video_dir') else None,
         constants.VIDEO_CACHE_DIR / V2S / f'{DATASET}_s8_squash',
         V2_CACHE / 'video' / V2S / f'{DATASET}_s8_squash']
t2_video = next((c for c in cands if c and (c / constants.VIDEO_MANIFEST_FILENAME).is_file()), None)
assert t2_video is not None, f'cannot find the T2 V2-S cache manifest; looked in {cands}'
T2_VIDEO_MANIFEST = json.loads((t2_video / constants.VIDEO_MANIFEST_FILENAME).read_text())
assert T2_VIDEO_MANIFEST['encoder'] == V2S and T2_VIDEO_MANIFEST['stride'] == STRIDE, T2_VIDEO_MANIFEST
print(f'T2 bake: train_ids_sha1 {t2_bake["train_ids_sha1"][:12]}, width {t2_bake["width"]}; '
      f'T2 V2-S manifest from {t2_video}')

## 2. Kaggle frames → a numeric-order symlink farm (T4, T9(c) part 1)

TAD's frame names are not zero-padded (`10.jpg` sorts before `2.jpg`, TAD_SETUP §4.3). The farm links every frame of
the 100 test videos under `{numeric index:06d}{ext}`, so lexicographic order **is** temporal order; the Kaggle copy is
never renamed. Gate: every video's frame count `N` gives `⌈N/8⌉ == len(TAD_ncc)` — the frames T0 was measured on.

In [ ]:
import kagglehub

FARM = Path('/content/tad_farm')
IMG = {'.jpg', '.jpeg', '.png', '.bmp'}
GATE_JSON = OUT / 'REPORTS' / 'tadz_frame_gate.json'
need_farm = not complete(TAD_V2S, test_ids) or not GATE_JSON.exists()

if need_farm and not (FARM / 'DONE').exists():
    try:
        from google.colab import userdata          # only needed if the dataset ever stops being public
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
            os.environ[k] = os.environ.get(k) or userdata.get(k)
    except Exception as exc:  # noqa: BLE001 -- secrets are optional
        print('no Kaggle secrets (fine for a public dataset):', type(exc).__name__)
    t0 = time.time()
    root = Path(kagglehub.dataset_download(KAGGLE))
    print(f'Kaggle dataset at {root} ({time.time() - t0:.0f}s)')

    wanted, found = set(test_ids), {}
    for dirpath, _, files in os.walk(root):
        if not any(Path(f).suffix.lower() in IMG for f in files):
            continue
        vid = re.sub(r'\.(mp4|avi|mov|mkv)$', '', Path(dirpath).name, flags=re.I)
        if vid in wanted:
            assert vid not in found, f'{vid} appears twice: {found[vid]} and {dirpath}'
            found[vid] = Path(dirpath)
    lost = sorted(wanted - set(found))
    assert not lost, f'{len(lost)} test videos have no frame folder in the Kaggle copy: {lost[:5]}'

    def key(p):
        digits = re.findall(r'\d+', p.stem)
        assert digits, f'no frame index in {p}'
        return int(digits[-1])

    if FARM.exists():
        shutil.rmtree(FARM)
    counts = {}
    for vid, src in sorted(found.items()):
        frames = sorted((p for p in src.iterdir() if p.suffix.lower() in IMG), key=key)
        idx = [key(p) for p in frames]
        assert len(set(idx)) == len(idx), f'{vid}: duplicate frame indices'
        dst = FARM / vid
        dst.mkdir(parents=True)
        for i, p in enumerate(frames):
            (dst / f'{i:06d}{p.suffix.lower()}').symlink_to(p)
        counts[vid] = len(frames)
    bad = {v: (n, math.ceil(n / STRIDE), len(np.load(TAD_NCC / f'{v}.npy', mmap_mode='r')))
           for v, n in counts.items() if math.ceil(n / STRIDE) != len(np.load(TAD_NCC / f'{v}.npy', mmap_mode='r'))}
    assert not bad, f'T9(c): Kaggle frame counts disagree with TAD_ncc (N, ceil(N/8), rows): {list(bad.items())[:5]}'
    (FARM / 'DONE').write_text(json.dumps(counts))
    print(f'farm: {len(counts)} videos, {sum(counts.values())} frames; ceil(N/8) == TAD_ncc rows on all')
else:
    print('farm not needed (V2-S on Drive and frame gate passed) or already built -- skipped')

## 3. T9(c) part 2 — frame identity: CLIP s8 re-extracted from the farm vs `TAD_ncc`

Five test videos, `no_center_crop`, stride 8. Same rows and per-row cosine ≥ 0.999, or the Kaggle frames are not
the frames T0 (and A0) were measured on — stop.

In [ ]:
if GATE_JSON.exists():
    print('frame gate already passed:', json.loads(GATE_JSON.read_text()))
else:
    check_ids = test_ids[::len(test_ids) // constants.V2_TADZ_FRAME_GATE_IDS][:constants.V2_TADZ_FRAME_GATE_IDS]
    (LOCAL / 'gate_ids.txt').write_text('\n'.join(check_ids) + '\n')
    run([sys.executable, '-m', 'core.tools.extract_clip_features', '--frames-dir', FARM,
         '--ids-file', LOCAL / 'gate_ids.txt', '--dataset', constants.TAD_DATASET, '--stride', STRIDE,
         '--batch-size', 64, '--device', 'auto', '--no-center-crop', '--output-dir', LOCAL / 'clip_gate'],
        log=LOCAL / 'clip_gate.log')
    result = {}
    for v in check_ids:
        a = np.load(LOCAL / 'clip_gate' / f'{v}.npy').astype(np.float64)
        b = np.load(TAD_NCC / f'{v}.npy').astype(np.float64)
        assert a.shape == b.shape, f'{v}: {a.shape} vs {b.shape}'
        cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
        result[v] = float(cos.min())
    print({v: round(c, 6) for v, c in result.items()})
    assert min(result.values()) >= constants.V2_TADZ_FRAME_GATE_COS, 'T9(c): the Kaggle frames are not T0\'s frames -- stop'
    GATE_JSON.parent.mkdir(parents=True, exist_ok=True)
    GATE_JSON.write_text(json.dumps({'min_cos': result, 'ids': check_ids}, indent=2))
    print('frame gate PASS')

## 4. VideoMAE V2-S at stride 8 on the 100 test videos (T4) → Drive

In [ ]:
if complete(TAD_V2S, test_ids):
    print(f'{TAD_V2S} already complete -- not re-extracted')
else:
    local_v2s = LOCAL / 'v2s'
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.extract_video_features', '--frames-dir', FARM,
         '--ids-file', LOCAL / 'test_ids.txt', '--dataset', constants.TAD_DATASET, '--encoder', V2S,
         '--stride', STRIDE, '--batch-size', 32, '--device', 'auto', '--output-dir', local_v2s],
        log=LOCAL / 'v2s.log')
    print(f'V2-S extracted in {(time.time() - t0) / 60:.1f} min')
    publish(local_v2s, TAD_V2S)

man = json.loads((TAD_V2S / constants.VIDEO_MANIFEST_FILENAME).read_text())
diff = sorted(k for k in set(man) | set(T2_VIDEO_MANIFEST) if man.get(k) != T2_VIDEO_MANIFEST.get(k))
assert not diff, f'T4: TAD V2-S manifest differs from T2\'s on {diff}'
bad = [v for v in test_ids
       if len(np.load(TAD_V2S / f'{v}.npy', mmap_mode='r')) != len(np.load(TAD_NCC / f'{v}.npy', mmap_mode='r'))]
assert not bad, f'T9(f): V2-S rows != CLIP rows on {bad[:5]}'
print('V2-S: manifest == T2\'s on every key; rows == TAD_ncc rows on all 100')

## 5. Bake A3's input on TAD with the T2 statistics (T3) → Drive; T9(f)

In [ ]:
L_NCC = stage_ids(TAD_NCC, LOCAL / 'stage' / 'TAD_ncc', test_ids)
if complete(TAD_BAKE, test_ids):
    print(f'{TAD_BAKE} already baked -- not re-baked')
else:
    stats = LOCAL / 'stage' / 't2_stats'
    stats.mkdir(parents=True, exist_ok=True)
    for f in (constants.V2_INPUT_MANIFEST_FILENAME, 'v2_input_stats.npz'):
        shutil.copy2(T2_BAKE / f, stats / f)
    L_V2S = stage_ids(TAD_V2S, LOCAL / 'stage' / 'TAD_v2s', test_ids)
    run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', stats,
         '--clip-dir', L_NCC, '--video-dir', L_V2S, '--ids-file', LOCAL / 'test_ids.txt', '--stride', 1,
         '--out-dir', LOCAL / 'bake'], log=LOCAL / 'bake.log')
    publish(LOCAL / 'bake', TAD_BAKE)

bake = json.loads((TAD_BAKE / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert (bake['crn'], bake['motion']) == (CRN, V2S), bake
assert bake['train_ids_sha1'] == t2_bake['train_ids_sha1'], 'T9(f): TAD bake used other T2 stats'
assert bake['width'] == t2_bake['width'] == 512 + constants.VIDEOMAE_ARCH[V2S][0], bake['width']
assert bake['baked_ids'] == len(test_ids), bake['baked_ids']
L_BAKE = stage_ids(TAD_BAKE, LOCAL / 'stage' / 'TAD_bake', test_ids)
print(f'bake OK: arm {bake["arm"]}, width {bake["width"]}, T2 stats {bake["train_ids_sha1"][:12]}')

## 6. Score — T0, then {A0, A3, A3-ign} × 5 seeds (`core.evaluate`, T2)

`--set data.dataset=TAD`: TAD's own definition and verbalizer, exactly as Gate T0. `--score-norm auto` resolves to
raw (TAD has normals). Each run lands in `v2_tadz/<arm>/s<seed>/{results.json, scores/}`.

In [ ]:
L_TAD = LOCAL / 'stage' / 'TAD'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never the frame archive
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)

jobs = [('T0', None, ['--baseline-ckpt', T0_CKPT, '--set', 'kip.enabled=false'], L_NCC)]
for arm in ARMS:
    for seed in SEEDS:
        jobs.append((arm, seed, ['--ckpt', run_dir(arm, seed) / 'checkpoint_last.pt'],
                     L_NCC if arm == 'A0' else L_BAKE))
for arm, seed, ck, clip_dir in jobs:
    rel = Path(arm) if seed is None else Path(arm) / f's{seed}'
    if (OUT / rel / 'results.json').exists():
        print(f'{rel}: already scored -- not re-scored')
        continue
    local = LOCAL / 'runs' / rel
    if local.exists():
        shutil.rmtree(local)
    run([sys.executable, '-m', 'core.evaluate', *ck, '--set', f'data.dataset={constants.TAD_DATASET}',
         '--data-dir', L_TAD, '--clip-dir', clip_dir, '--score-norm', 'auto', '--save-scores',
         '--output-dir', local], log=LOCAL / f'{str(rel).replace("/", "_")}.log')
    publish(local, OUT / rel)
    r = json.loads((OUT / rel / 'results.json').read_text())
    print(f'{rel}: macro {r["auc_macro"]:.4f} over {r["auc_macro_videos"]} | micro {r["auc"]:.4f}')
t0 = json.loads((OUT / 'T0' / 'results.json').read_text())
print(f'T0 re-scored: micro {t0["auc"]:.4f} (registered {constants.V2_TADZ_T0_MICRO}), macro {t0["auc_macro"]:.4f}')

## 7. Read-out (T9 gates first, then T5–T7) — no GPU

In [ ]:
run([sys.executable, '-m', 'core.tools.tadz_readout', '--root', OUT, '--t2-v2-dir', V2_DATA])
print((OUT / 'REPORTS' / 'tadz_readout.md').read_text())

## 8. Record — then **stop**

Paste `v2_tadz/REPORTS/tadz_readout.md` back to Claude (and `tadz_frame_gate.json` if asked). The sentence is fixed by
§26 T5/T6; `B` = A3-ign stands whatever it says (T1). Nothing is re-run, re-scored at another stride, or tuned after
reading it (T8). Whether TAD enters the thesis report is decided by the user and the advisor (T10).